# 论文喘息算法验证

## tl;dr

- 现有数据只对论文的频谱判据提供**方向性支持**，不能证明完整算法已经正确。
- 主分析按 25 Hz 实际采样率、90 秒非重叠窗、合加速度幅值和矩形窗复算。去除跨文件重复前缀后，喘息文件 56 个窗口中 33 个满足 `F_1_2 > F_2_3`（58.9%）；反刍对照 13 个窗口中 12 个被排除（92.3%）。
- 当前固件使用 25 Hz、6 秒时域正弦检测，不是论文的 90 秒 FFT 谐波比算法；两者结果不能互相替代。


## Context & Methods

论文明确给出的规则是：在 90 秒窗内，分别计算 1–2 Hz 与 2–3 Hz 频带的“峰值/频带均值” `F_1_2` 与 `F_2_3`；若一个窗口先被判为反刍且 `F_1_2 > F_2_3`，则改判为热应激/快速喘息。

### Key Assumptions

- CSV 本身没有采样率字段；采用项目 PC runner 与算法代码声明的 25 Hz，而不是论文设备的 10 Hz。
- 主解释对 `sqrt(x^2+y^2+z^2)` 去均值后做 FFT；频带使用半开区间 `[1,2)` 和 `[2,3)`，避免 2 Hz 重复计入。
- 文件名仅作为记录级标签。没有逐窗人工呼吸率、温湿度、THI 或喘息起止区间，因此下文的通过率不是生产敏感度/特异度。


In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

validation_dir = Path('Doc/Paper/validation').resolve()
analysis_script = validation_dir / 'validate_panting_algorithm.py'
result = subprocess.run([sys.executable, str(analysis_script)], check=True, capture_output=True, text=True)
print(result.stdout.strip())


output_dir=/Users/gally/Library/CloudStorage/Dropbox/Git/GitHub/RFID_CC1101_433MHz_V2/Doc/Paper/validation/output
duplicate_prefix_rows=15536
primary: panting_pass=33/56 (58.9%), rumination_rejected=12/13 (92.3%)
firmware_comparison=completed


## Data

四个文件共包含三个“喘息”记录和一个“反刍”对照。质量检查重点是采样范围、时间字段、初始化行、文件内重复，以及两个 2025-08-20 文件之间的跨文件重叠。


In [2]:
output_dir = validation_dir / 'output'
quality = pd.read_csv(output_dir / 'data_quality_summary.csv')
quality_columns = [
    'recording', 'raw_rows', 'analysis_rows_after_cross_file_dedup',
    'deduplicated_prefix_rows', 'null_xyz_rows',
    'out_of_adxl362_12bit_range_rows', 'timestamp_column_present'
]
print(quality[quality_columns].to_string(index=False))


          recording  raw_rows  analysis_rows_after_cross_file_dedup  deduplicated_prefix_rows  null_xyz_rows  out_of_adxl362_12bit_range_rows  timestamp_column_present
               反刍对照     30558                                 30558                         0              0                                0                     False
喘息 2025-08-19 16:22     19721                                 19720                         0              0                                0                     False
喘息 2025-08-20 16:44     77846                                 77845                         0              0                                0                     False
喘息 2025-08-20 17:55     47425                                 31888                     15536              0                                0                     False


## Results

主判据能较好排除反刍对照，但在三个喘息文件之间表现不稳定。特别是 2025-08-19 记录只有 1/8 个 90 秒窗通过，而 2025-08-20 17:55 记录有 12/14 个通过。这种文件间异质性说明不能用一个总体百分比掩盖采集姿态、行为混合或信号解释差异。


In [3]:
summary = json.loads((output_dir / 'validation_summary.json').read_text(encoding='utf-8'))
primary = summary['primary_results']
print(
    f"喘息窗口通过：{primary['panting_windows_passing']}/{primary['panting_windows']} "
    f"({primary['panting_window_pass_rate']:.1%})\n"
    f"反刍窗口排除：{primary['rumination_windows_rejected']}/{primary['rumination_windows']} "
    f"({primary['rumination_window_rejection_rate']:.1%})\n"
    f"描述性平衡率：{primary['balanced_descriptive_rate']:.1%}"
)
recordings = pd.read_csv(output_dir / 'recording_summary.csv')
display_columns = [
    'recording', 'expected_behavior', 'windows', 'criterion_passes',
    'criterion_pass_rate', 'median_spectral_ratio', 'median_peak_1_2_bpm'
]
print('\n' + recordings[display_columns].round(3).to_string(index=False))


喘息窗口通过：33/56 (58.9%)
反刍窗口排除：12/13 (92.3%)
描述性平衡率：75.6%

          recording expected_behavior  windows  criterion_passes  criterion_pass_rate  median_spectral_ratio  median_peak_1_2_bpm
               反刍对照        rumination       13                 1                0.077                  0.771               76.667
喘息 2025-08-19 16:22           panting        8                 1                0.125                  0.632              114.667
喘息 2025-08-20 16:44           panting       34                20                0.588                  1.056               99.667
喘息 2025-08-20 17:55           panting       14                12                0.857                  1.209              114.000


## Robustness Checks

结果依赖论文未说明的处理选择。Hann 窗把反刍排除率从 92.3% 提高到 100%，但喘息通过率不变；把三轴公式误解为一阶差分幅值会把反刍排除率降至 7.7%；把 25 Hz 数据误按论文设备的 10 Hz 处理，描述性平衡率降至接近随机的 49.4%。


In [4]:
sensitivity = pd.read_csv(output_dir / 'sensitivity_analysis.csv')
sensitivity_columns = [
    'check', 'detail', 'panting_windows', 'panting_window_pass_rate',
    'rumination_windows', 'rumination_window_rejection_rate',
    'balanced_descriptive_rate'
]
print(sensitivity[sensitivity_columns].round(3).to_string(index=False))


                check                                                      detail  panting_windows  panting_window_pass_rate  rumination_windows  rumination_window_rejection_rate  balanced_descriptive_rate
              primary 25 Hz, 90 s, raw magnitude, rectangular taper, deduplicated               56                     0.589                  13                             0.923                      0.756
                taper                                                  Hann taper               56                     0.589                  13                             1.000                      0.795
signal_interpretation                                  dominant peak-to-peak axis               56                     0.607                  13                             0.846                      0.727
signal_interpretation                                           dominant RMS axis               56                     0.554                  13                             0.8

## Current Firmware Is a Different Algorithm

PC runner 直接编译当前 `Core/Src/adxl362_behavior.c`。它在反刍对照中没有输出 `breath`，但三个喘息文件的样本级 `breath` 占比分别约为 85.1%、30.8% 和去重后的 74.0%。这只能说明当前固件在文件级标签上有响应，不能验证其逐秒准确率，也不能作为论文 FFT 规则的复现结果。


In [5]:
firmware = pd.read_csv(output_dir / 'firmware_comparison.csv')
firmware_columns = [
    'recording', 'expected_behavior', 'processed_samples_after_dedup',
    'breath_samples', 'breath_sample_rate'
]
print(firmware[firmware_columns].round(3).to_string(index=False))


          recording expected_behavior  processed_samples_after_dedup  breath_samples  breath_sample_rate
               反刍对照        rumination                          30450               0               0.000
喘息 2025-08-19 16:22           panting                          19650           16725               0.851
喘息 2025-08-20 16:44           panting                          77700           23950               0.308
喘息 2025-08-20 17:55           panting                          31863           23563               0.740


## Takeaways

1. `F_1_2 > F_2_3` 在这批数据中表现出方向性区分能力，尤其能排除提供的反刍对照。
2. 由于喘息文件通过率只有 58.9%，且三个文件间从 12.5% 到 85.7% 波动，现有证据不足以宣称算法正确或可直接上线。
3. 完整验证需要逐窗人工呼吸率真值、牛只/姿态标识、温湿度/THI、更多反刍及其他活动负样本，并明确论文第 8 步分类器和 FFT 预处理细节。
4. 若目标是验证当前固件，应另做 6 秒级标注评估；论文验证与固件验证必须分开报告。
